# NTU CCTV-Fights Stage 1

## Main workflows

Set `WORKFLOW` in the configuration cell:

- `index`: build the video index and write final ground-truth frame-label CSVs.
- `train`: build labels, train MobileNetV2+GRU, save checkpoints, validate, and export sample predictions.
- `posttest`: load a checkpoint, run full-video inference, threshold sweeps, per-video evaluation, and overlays.
- `none`: define functions only, then use the manual cells.

## Local storage option

For faster Colab runs, set `COPY_DRIVE_DATA_TO_LOCAL = True`. The dataset is copied from Drive to `/content/ntu_cctv_stage1`, while outputs can still be saved back to Drive with `SAVE_OUTPUTS_TO_DRIVE = True`.


In [ ]:
# Colab setup
!pip install -q opencv-python scikit-learn matplotlib pandas tqdm

%matplotlib inline


## Core pipeline definitions

The following cells contain the merged, de-duplicated implementation.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - fallback when tqdm is unavailable
    def tqdm(x, **kwargs):
        return x

# Torchvision is imported lazily so the script can still build indexes or show
# --help in environments where torch/torchvision versions are temporarily mismatched.
models = None
transforms = None


def require_torchvision():
    global models, transforms
    if models is not None and transforms is not None:
        return models, transforms

    try:
        from torchvision import models as _models, transforms as _transforms
    except Exception as exc:
        raise ImportError(
            "Could not import torchvision. Install compatible torch/torchvision versions, e.g. "
            "`pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121` "
        ) from exc

    models = _models
    transforms = _transforms
    return models, transforms


In [ ]:
# ----------------------------
# Configuration
# ----------------------------

DEFAULT_VIDEO_SUBDIRS = [
    "original_format-001-200",
    "original_format-201-400",
    "original_format-401-600",
    "original_format-601-800",
    "original_format-801-1000",
]

VIDEO_EXTS = [".mp4", ".avi", ".mov", ".webm", ".mkv"]


@dataclass
class Config:
    command: str
    base_dir: Path
    video_dirs: List[Path]
    gt_json: Optional[Path]
    output_dir: Path
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Label/index settings.
    max_videos: Optional[int] = None
    use_json_subsets: bool = True
    label_length_source: str = "video"  # "video" or "json"
    label_fps_source: str = "json"      # "json" or "video"
    min_valid_fps: float = 5.0
    max_valid_fps: float = 120.0
    verify_decoded_frame_count: bool = True
    sequential_read_fallback: bool = True

    # Model/training.
    img_size: int = 160
    window_size: int = 32
    train_stride: int = 64
    infer_stride: int = 16
    batch_size: int = 8
    num_workers: int = 0
    max_train_windows_per_video: Optional[int] = 8
    max_val_windows_per_video: Optional[int] = 4
    max_total_train_windows: Optional[int] = 5000
    max_total_val_windows: Optional[int] = 1500
    num_epochs: int = 5
    lr: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.5
    grad_clip_norm: Optional[float] = 1.0
    patience: int = 3
    freeze_backbone: bool = True
    pretrained_backbone: bool = True
    use_balanced_window_sampler: bool = False
    max_pos_weight: float = 1.0
    save_each_epoch: bool = True
    save_optimizer_state: bool = True

    # Temporal post-processing.
    train_eval_threshold: float = 0.5
    post_smooth_k: int = 5
    post_high_thresh: float = 0.60
    post_low_thresh: float = 0.40
    post_max_gap: int = 5
    post_min_len: int = 8
    post_pre_pad: int = 5
    post_post_pad: int = 5

    # Evaluation/visualization.
    max_full_eval_videos: Optional[int] = 10
    label_video_max_frames: Optional[int] = 600
    label_video_resize_width: Optional[int] = 720
    export_gt_overlay: bool = True
    export_sample_overlay: bool = True
    run_gradcam: bool = False

    # Post-test specific.
    checkpoint: Optional[Path] = None
    raw_threshold: float = 0.50
    thresholds_to_plot: List[float] = field(default_factory=lambda: [0.30, 0.40, 0.50, 0.60, 0.70])
    global_thresholds: List[float] = field(default_factory=lambda: [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)])
    eval_split: str = "validation"  # "training", "validation", "testing", or "all"
    max_eval_videos: Optional[int] = 20
    sample_video_id: Optional[str] = None


@dataclass
class RuntimePaths:
    label_dir: Path
    checkpoint_dir: Path
    epoch_ckpt_dir: Path
    pred_dir: Path
    plot_dir: Path
    label_video_dir: Path
    gradcam_dir: Path


def make_runtime_paths(cfg: Config) -> RuntimePaths:
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    paths = RuntimePaths(
        label_dir=cfg.output_dir / "frame_labels",
        checkpoint_dir=cfg.output_dir / "checkpoints",
        epoch_ckpt_dir=cfg.output_dir / "checkpoints" / "epochs",
        pred_dir=cfg.output_dir / "predictions",
        plot_dir=cfg.output_dir / "plots",
        label_video_dir=cfg.output_dir / "label_videos",
        gradcam_dir=cfg.output_dir / "gradcam",
    )
    for p in [
        paths.label_dir,
        paths.checkpoint_dir,
        paths.epoch_ckpt_dir,
        paths.pred_dir,
        paths.plot_dir,
        paths.label_video_dir,
        paths.gradcam_dir,
    ]:
        p.mkdir(parents=True, exist_ok=True)
    return paths


In [ ]:
# ----------------------------
# General utilities
# ----------------------------

def maybe_mount_drive(base_dir: Path, mount_drive: bool) -> None:
    """Mount Google Drive only when running in Colab and requested/needed."""
    should_try = mount_drive or (str(base_dir).startswith("/content/drive") and not base_dir.exists())
    if not should_try:
        return

    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[WARN] Could not mount Google Drive automatically: {exc}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def natural_key(s: Any) -> List[Any]:
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


def first_existing(paths: Iterable[Path]) -> Optional[Path]:
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def default_gt_candidates(cfg: Config) -> List[Path]:
    candidates: List[Path] = []
    if cfg.gt_json is not None:
        candidates.append(Path(cfg.gt_json))
    candidates.append(cfg.base_dir / "groundtruth.json")
    for d in cfg.video_dirs:
        candidates.append(Path(d) / "groundtruth.json")
    return candidates


def load_groundtruth_json(cfg: Config) -> Tuple[Path, Dict[str, Any]]:
    p = first_existing(default_gt_candidates(cfg))
    if p is None:
        raise FileNotFoundError(
            "Could not find groundtruth.json. Pass --gt-json, or place it at "
            "<base-dir>/groundtruth.json or inside one of the video folders."
        )
    print("Using groundtruth:", p)
    with open(p, "r", encoding="utf-8") as f:
        gt = json.load(f)
    if "database" not in gt:
        raise KeyError("groundtruth.json does not contain a top-level 'database' key.")
    print("GT version:", gt.get("version"))
    print("Total entries in JSON:", len(gt["database"]))
    print("Example keys:", list(gt["database"].keys())[:5])
    return p, gt


def build_video_map(video_dirs: Sequence[Path]) -> Tuple[Dict[str, Path], List[Path]]:
    """Index videos across one or more folders by file stem, e.g. fight_0001."""
    video_map: Dict[str, Path] = {}
    video_files: List[Path] = []
    duplicate_stems: List[str] = []

    for video_dir in map(Path, video_dirs):
        if not video_dir.exists():
            print(f"[WARN] VIDEO_DIR does not exist, skipping: {video_dir}")
            continue

        found_here = 0
        for p in video_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
                found_here += 1
                video_files.append(p)

                if p.stem in video_map:
                    duplicate_stems.append(p.stem)
                    continue

                video_map[p.stem] = p

        print(f"Found {found_here} videos under {video_dir}")

    video_files = sorted(video_files, key=lambda p: natural_key(p.stem))

    print(f"Total video files found: {len(video_files)}")
    print(f"Unique video stems indexed: {len(video_map)}")

    if duplicate_stems:
        print("[WARN] Duplicate video stems found. Keeping the first occurrence.")
        print("Duplicate examples:", sorted(set(duplicate_stems), key=natural_key)[:10])

    for p in video_files[:5]:
        print(" sample:", p.name)

    return video_map, video_files


def count_decodable_frames(video_path: Path) -> Optional[int]:
    """Sequentially count frames that OpenCV can actually decode."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    count = 0
    while True:
        ok, _ = cap.read()
        if not ok:
            break
        count += 1

    cap.release()
    return count


def get_video_metadata(video_path: Path, cfg: Config) -> Dict[str, Optional[float]]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video_fps": None,
            "video_frame_count": None,
            "reported_frame_count": None,
            "decoded_frame_count": None,
            "width": None,
            "height": None,
        }

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    reported_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps is None or fps <= 0 or np.isnan(fps):
        fps = None

    decoded_frame_count = None
    if cfg.verify_decoded_frame_count:
        decoded_frame_count = count_decodable_frames(video_path)

    video_frame_count = decoded_frame_count if decoded_frame_count is not None else reported_frame_count

    return {
        "video_fps": fps,
        "video_frame_count": video_frame_count,
        "reported_frame_count": reported_frame_count,
        "decoded_frame_count": decoded_frame_count,
        "width": width,
        "height": height,
    }


def is_sane_fps(fps: Optional[float], cfg: Config) -> bool:
    if fps is None:
        return False
    try:
        fps = float(fps)
    except Exception:
        return False
    if np.isnan(fps):
        return False
    return cfg.min_valid_fps <= fps <= cfg.max_valid_fps


def segment_json_to_frame_labels(
    info: Dict[str, Any],
    video_meta: Optional[Dict[str, Any]],
    cfg: Config,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Convert NTU-CCTV JSON time intervals to per-frame 0/1 labels.
    """
    json_nb_frames = int(info["nb_frames"])
    json_fps = float(info["frame_rate"])

    video_frame_count = None
    video_fps = None

    if video_meta is not None:
        video_frame_count = video_meta.get("video_frame_count")
        video_fps = video_meta.get("video_fps")

    if cfg.label_length_source == "video" and video_frame_count is not None:
        nb_frames = int(video_frame_count)
        length_source_used = "video"
    else:
        nb_frames = json_nb_frames
        length_source_used = "json"

    if cfg.label_fps_source == "video" and is_sane_fps(video_fps, cfg):
        fps = float(video_fps)
        fps_source_used = "video"
    else:
        fps = json_fps
        fps_source_used = "json"

    labels = np.zeros(nb_frames, dtype=np.int32)

    for ann in info.get("annotations", []):
        if ann.get("label", "").lower() != "fight":
            continue

        start_sec, end_sec = ann["segment"]

        start_frame = int(round(float(start_sec) * fps))
        end_frame = int(round(float(end_sec) * fps))

        start_frame = max(0, min(nb_frames - 1, start_frame))
        end_frame = max(0, min(nb_frames - 1, end_frame))

        if end_frame >= start_frame:
            labels[start_frame:end_frame + 1] = 1

    meta_out = {
        "label_length_source_used": length_source_used,
        "label_fps_source_used": fps_source_used,
        "label_fps_used": float(fps),
        "label_nb_frames_used": int(nb_frames),
        "video_fps_sane": is_sane_fps(video_fps, cfg),
    }

    return labels, meta_out


def get_positive_ranges(labels: Sequence[int]) -> List[Tuple[int, int]]:
    labels_np = np.asarray(labels).astype(int)
    ranges: List[Tuple[int, int]] = []
    start = None

    for i, v in enumerate(labels_np):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            ranges.append((start, i - 1))
            start = None

    if start is not None:
        ranges.append((start, len(labels_np) - 1))

    return ranges


def save_labels_csv(labels: np.ndarray, label_path: Path) -> None:
    label_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(label_path, labels.astype(np.int32), fmt="%d", delimiter=",")


def load_label_csv(label_path: Path | str) -> np.ndarray:
    labels = np.loadtxt(label_path, delimiter=",")
    labels = np.asarray(labels).astype(np.int32).reshape(-1)
    return labels


def build_index_from_json(
    database: Dict[str, Any],
    video_map: Dict[str, Path],
    cfg: Config,
    paths: RuntimePaths,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    matched_ids = [video_id for video_id in sorted(database.keys(), key=natural_key) if video_id in video_map]

    if cfg.max_videos is not None:
        matched_ids = matched_ids[: int(cfg.max_videos)]

    print("Matched JSON entries with videos:", len(matched_ids))

    for idx, video_id in enumerate(matched_ids):
        if idx < 5 or (idx + 1) % 50 == 0 or idx == len(matched_ids) - 1:
            print(f"[{idx + 1}/{len(matched_ids)}] {video_id}")

        info = database[video_id]
        video_path = video_map[video_id]
        meta = get_video_metadata(video_path, cfg)

        labels, label_meta = segment_json_to_frame_labels(info, video_meta=meta, cfg=cfg)

        label_path = paths.label_dir / f"{video_id}.csv"
        save_labels_csv(labels, label_path)

        json_nb_frames = int(info["nb_frames"])
        json_fps = float(info["frame_rate"])
        video_frame_count = meta["video_frame_count"]
        reported_frame_count = meta.get("reported_frame_count")
        decoded_frame_count = meta.get("decoded_frame_count")
        video_fps = meta["video_fps"]

        length_diff = None
        if video_frame_count is not None:
            length_diff = int(len(labels) - int(video_frame_count))

        ranges = get_positive_ranges(labels)

        rows.append({
            "video_id": video_id,
            "subset": info.get("subset", "unknown"),
            "source": info.get("source", "unknown"),
            "video_path": str(video_path),
            "label_path": str(label_path),
            "json_nb_frames": json_nb_frames,
            "json_fps": json_fps,
            "video_frame_count": video_frame_count,
            "reported_frame_count": reported_frame_count,
            "decoded_frame_count": decoded_frame_count,
            "video_fps": video_fps,
            "label_fps_used": label_meta["label_fps_used"],
            "label_fps_source_used": label_meta["label_fps_source_used"],
            "label_length_source_used": label_meta["label_length_source_used"],
            "video_fps_sane": label_meta["video_fps_sane"],
            "width": meta["width"],
            "height": meta["height"],
            "label_len": len(labels),
            "length_diff_vs_video": length_diff,
            "fight_frames": int(labels.sum()),
            "nonfight_frames": int(len(labels) - labels.sum()),
            "fight_ratio": float(labels.mean()) if len(labels) else 0.0,
            "num_segments": len(ranges),
            "segments_frame": ranges,
            "segments_sec_from_labels": [(s / label_meta["label_fps_used"], e / label_meta["label_fps_used"]) for s, e in ranges],
        })

    index_df = pd.DataFrame(rows)
    if len(index_df) == 0:
        raise RuntimeError("No videos matched between VIDEO_DIRS and groundtruth.json.")

    index_path = cfg.output_dir / "index.csv"
    index_df.to_csv(index_path, index=False)
    print("Saved index:", index_path)

    print("\nIndex preview:")
    preview_cols = [
        "video_id", "subset", "source", "label_len", "video_frame_count",
        "reported_frame_count", "decoded_frame_count", "json_nb_frames", "video_fps", "json_fps",
        "label_fps_used", "label_fps_source_used", "video_fps_sane",
        "fight_frames", "nonfight_frames", "fight_ratio", "num_segments",
    ]
    print(index_df[[c for c in preview_cols if c in index_df.columns]].head(20).to_string(index=False))

    print("\nTotal matched videos:", len(index_df))
    print("Subsets:")
    print(index_df["subset"].value_counts(dropna=False))
    print()
    print("Total fight frames:", int(index_df["fight_frames"].sum()))
    print("Total non-fight frames:", int(index_df["nonfight_frames"].sum()))
    print("Overall fight ratio:", index_df["fight_frames"].sum() / max(index_df["label_len"].sum(), 1))

    mismatch_df = index_df[
        index_df["length_diff_vs_video"].notna() &
        (index_df["length_diff_vs_video"].abs() > 5)
    ]
    print("\nVideos with label/video length mismatch > 5 frames:", len(mismatch_df))
    if len(mismatch_df) > 0:
        print(mismatch_df[[
            "video_id", "label_len", "video_frame_count", "reported_frame_count",
            "decoded_frame_count", "json_nb_frames", "length_diff_vs_video",
        ]].head(20).to_string(index=False))

    bad_fps_df = index_df[
        index_df["video_fps"].notna() &
        (
            (index_df["video_fps"] < cfg.min_valid_fps) |
            (index_df["video_fps"] > cfg.max_valid_fps) |
            ((index_df["video_fps"] - index_df["json_fps"]).abs() > 5)
        )
    ]

    print("\nVideos with suspicious OpenCV FPS:", len(bad_fps_df))
    if len(bad_fps_df) > 0:
        print(bad_fps_df[[
            "video_id", "video_fps", "json_fps", "label_fps_used",
            "label_fps_source_used", "fight_frames", "fight_ratio",
            "segments_frame", "segments_sec_from_labels",
        ]].head(20).to_string(index=False))

    return index_df


In [ ]:
# ----------------------------
# Video overlay utilities
# ----------------------------

def get_video_fps(video_path: Path, cfg: Config, default_fps: float = 25.0, fallback_fps: Optional[float] = None) -> float:
    meta = get_video_metadata(Path(video_path), cfg)
    video_fps = meta.get("video_fps")

    if is_sane_fps(video_fps, cfg):
        return float(video_fps)

    if fallback_fps is not None and is_sane_fps(fallback_fps, cfg):
        return float(fallback_fps)

    return float(default_fps)


def choose_label_video_range(labels: Sequence[int], max_frames: Optional[int] = 600, pre_context: int = 60) -> Tuple[int, int, str]:
    labels_np = np.asarray(labels).astype(int)
    n = len(labels_np)

    if max_frames is None or max_frames >= n:
        return 0, n, "full video"

    fight_idxs = np.where(labels_np == 1)[0]
    if len(fight_idxs) == 0:
        return 0, min(n, max_frames), "no fight label found; using beginning"

    first_fight = int(fight_idxs[0])
    start = max(0, first_fight - pre_context)
    end = min(n, start + max_frames)
    start = max(0, end - max_frames)
    return start, end, f"around first fight frame {first_fight}"


def draw_corner_label(
    frame_bgr: np.ndarray,
    frame_idx: int,
    gt_label: Optional[int] = None,
    pred_label: Optional[int] = None,
    pred_prob: Optional[float] = None,
    extra_text: Optional[str] = None,
) -> np.ndarray:
    lines = [f"Frame: {frame_idx}"]

    if gt_label is not None:
        gt_text = "FIGHT" if int(gt_label) == 1 else "NO FIGHT"
        lines.append(f"GT: {gt_text}")

    if pred_label is not None:
        pred_text = "FIGHT" if int(pred_label) == 1 else "NO FIGHT"
        if pred_prob is not None:
            lines.append(f"Pred: {pred_text}  p={float(pred_prob):.3f}")
        else:
            lines.append(f"Pred: {pred_text}")

    if extra_text is not None:
        lines.append(str(extra_text))

    h, w = frame_bgr.shape[:2]
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = max(0.55, min(w, h) / 900)
    thickness = max(1, int(round(min(w, h) / 350)))

    margin = 10
    line_height = int(28 * font_scale / 0.7)
    box_w = 0
    for line in lines:
        (tw, _), _ = cv2.getTextSize(line, font, font_scale, thickness)
        box_w = max(box_w, tw)

    box_h = margin * 2 + line_height * len(lines)
    box_w = box_w + margin * 2

    overlay = frame_bgr.copy()
    cv2.rectangle(overlay, (8, 8), (8 + box_w, 8 + box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.62, frame_bgr, 0.38, 0, frame_bgr)

    y = 8 + margin + int(18 * font_scale / 0.7)

    for line in lines:
        color = (255, 255, 255)

        if line.startswith("GT:"):
            color = (40, 40, 255) if "FIGHT" in line and "NO FIGHT" not in line else (80, 220, 80)

        if line.startswith("Pred:"):
            color = (0, 180, 255) if "FIGHT" in line and "NO FIGHT" not in line else (180, 255, 180)

        cv2.putText(frame_bgr, line, (18, y), font, font_scale, color, thickness, cv2.LINE_AA)
        y += line_height

    return frame_bgr


def export_labeled_video(
    video_path: Path,
    labels: Sequence[int],
    out_path: Path,
    cfg: Config,
    pred_labels: Optional[Sequence[int]] = None,
    pred_probs: Optional[Sequence[float]] = None,
    start_frame: Optional[int] = None,
    end_frame: Optional[int] = None,
    max_frames: Optional[int] = 600,
    resize_width: Optional[int] = 720,
    fps: Optional[float] = None,
    extra_text: Optional[str] = None,
) -> Path:
    video_path = Path(video_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    labels_np = np.asarray(labels).astype(int)

    if start_frame is None or end_frame is None:
        start_frame, end_frame, reason = choose_label_video_range(labels_np, max_frames=max_frames)
        print("Selected range:", start_frame, "to", end_frame, "|", reason)

    start_frame = int(max(0, start_frame))
    end_frame = int(min(len(labels_np), end_frame))

    if fps is None:
        fps = get_video_fps(video_path, cfg)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    writer = None
    written = 0

    for frame_idx in range(start_frame, end_frame):
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if resize_width is not None:
            h, w = frame_bgr.shape[:2]
            scale = resize_width / float(w)
            new_h = int(round(h * scale))
            frame_bgr = cv2.resize(frame_bgr, (resize_width, new_h), interpolation=cv2.INTER_AREA)

        pred = None if pred_labels is None or frame_idx >= len(pred_labels) else int(pred_labels[frame_idx])
        prob = None if pred_probs is None or frame_idx >= len(pred_probs) else float(pred_probs[frame_idx])
        gt = None if frame_idx >= len(labels_np) else int(labels_np[frame_idx])

        frame_bgr = draw_corner_label(
            frame_bgr,
            frame_idx=frame_idx,
            gt_label=gt,
            pred_label=pred,
            pred_prob=prob,
            extra_text=extra_text,
        )

        if writer is None:
            h, w = frame_bgr.shape[:2]
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))

        writer.write(frame_bgr)
        written += 1

    cap.release()
    if writer is not None:
        writer.release()

    print("Saved:", out_path)
    print("Frames written:", written)
    return out_path


def export_gt_overlay_sample(index_df: pd.DataFrame, cfg: Config, paths: RuntimePaths) -> Optional[Path]:
    with_fight = index_df[index_df["fight_frames"] > 0].copy()
    if len(with_fight) == 0:
        print("[WARN] No fight-labeled video found for GT overlay sample.")
        return None

    sample_row = with_fight.sort_values("fight_ratio", ascending=False).iloc[0]
    sample_video_id = sample_row["video_id"]
    sample_video_path = Path(sample_row["video_path"])
    sample_labels = load_label_csv(sample_row["label_path"])

    print("Sample video:", sample_video_id)
    print("Video path:", sample_video_path)
    print("GT positive ranges:", get_positive_ranges(sample_labels)[:10])
    print("GT positive ranges in seconds:", sample_row["segments_sec_from_labels"])

    gt_overlay_path = paths.label_video_dir / f"{sample_video_id}_gt_overlay.mp4"
    return export_labeled_video(
        sample_video_path,
        sample_labels,
        gt_overlay_path,
        cfg=cfg,
        max_frames=cfg.label_video_max_frames,
        resize_width=cfg.label_video_resize_width,
        fps=float(sample_row["label_fps_used"]),
        extra_text="GT label check",
    )


In [ ]:
# ----------------------------
# Splits, dataset, dataloaders
# ----------------------------

def make_splits(index_df: pd.DataFrame, cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = index_df.copy()

    if cfg.use_json_subsets:
        train_df = df[df["subset"].str.lower() == "training"].reset_index(drop=True)
        val_df = df[df["subset"].str.lower() == "validation"].reset_index(drop=True)
        test_df = df[df["subset"].str.lower() == "testing"].reset_index(drop=True)

        if len(train_df) > 0 and len(val_df) > 0:
            return train_df, val_df, test_df

        print("[WARN] JSON subsets were incomplete. Falling back to random split.")

    video_ids = df["video_id"].tolist()
    random.shuffle(video_ids)

    n = len(video_ids)
    n_train = max(1, int(0.8 * n))
    n_val = max(1, int(0.1 * n))

    train_ids = set(video_ids[:n_train])
    val_ids = set(video_ids[n_train:n_train + n_val])
    test_ids = set(video_ids[n_train + n_val:])

    train_df = df[df["video_id"].isin(train_ids)].reset_index(drop=True)
    val_df = df[df["video_id"].isin(val_ids)].reset_index(drop=True)
    test_df = df[df["video_id"].isin(test_ids)].reset_index(drop=True)

    return train_df, val_df, test_df


def make_image_transform(cfg: Config):
    _, transforms = require_torchvision()
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


class NTUCCTVWindowDataset(Dataset):
    def __init__(
        self,
        meta_df: pd.DataFrame,
        cfg: Config,
        window_size: int,
        stride: int,
        transform: Optional[Any] = None,
        max_windows_per_video: Optional[int] = None,
        max_total_windows: Optional[int] = None,
        split_name: str = "train",
        rng_seed: int = 42,
    ):
        self.meta_df = meta_df.reset_index(drop=True)
        self.cfg = cfg
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.transform = transform
        self.split_name = split_name

        rng = random.Random(rng_seed)
        self.samples: List[Dict[str, Any]] = []

        for _, row in self.meta_df.iterrows():
            labels = load_label_csv(row["label_path"])

            row_frame_count = row.get("video_frame_count", None)
            if pd.notna(row_frame_count):
                usable_len = min(len(labels), int(row_frame_count))
                labels = labels[:usable_len]
            else:
                usable_len = len(labels)

            if usable_len < self.window_size:
                continue

            video_samples = []

            for start in range(0, usable_len - self.window_size + 1, self.stride):
                end = start + self.window_size
                window_labels = labels[start:end]
                window_has_fight = int(window_labels.sum() > 0)

                video_samples.append({
                    "video_id": row["video_id"],
                    "video_path": row["video_path"],
                    "label_path": row["label_path"],
                    "start": start,
                    "end": end,
                    "window_has_fight": window_has_fight,
                })

            if max_windows_per_video is not None and len(video_samples) > max_windows_per_video:
                video_samples = self._balanced_sample_list(video_samples, max_windows_per_video, rng)

            self.samples.extend(video_samples)

        if max_total_windows is not None and len(self.samples) > max_total_windows:
            self.samples = self._balanced_sample_list(self.samples, max_total_windows, rng)

        labels_arr = np.array([s["window_has_fight"] for s in self.samples], dtype=np.int64) if self.samples else np.array([])
        counts = np.bincount(labels_arr, minlength=2) if len(labels_arr) else np.array([0, 0])

        print(
            f"[{split_name}] windows={len(self.samples)} "
            f"no-fight={counts[0]} fight={counts[1]} "
            f"approx decoded frames/epoch={len(self.samples) * self.window_size}"
        )

    @staticmethod
    def _balanced_sample_list(samples: List[Dict[str, Any]], max_items: int, rng: random.Random) -> List[Dict[str, Any]]:
        if len(samples) <= max_items:
            return samples

        pos = [i for i, s in enumerate(samples) if int(s["window_has_fight"]) == 1]
        neg = [i for i, s in enumerate(samples) if int(s["window_has_fight"]) == 0]

        chosen = set()

        if len(pos) > 0 and len(neg) > 0:
            n_pos = min(len(pos), max_items // 2)
            n_neg = min(len(neg), max_items - n_pos)
            chosen.update(rng.sample(pos, n_pos))
            chosen.update(rng.sample(neg, n_neg))
        else:
            pool = pos if len(pos) > 0 else neg
            chosen.update(rng.sample(pool, min(max_items, len(pool))))

        if len(chosen) < max_items:
            remaining = [i for i in range(len(samples)) if i not in chosen]
            chosen.update(rng.sample(remaining, min(max_items - len(chosen), len(remaining))))

        return [samples[i] for i in sorted(chosen)]

    def __len__(self) -> int:
        return len(self.samples)

    def _decode_frames_from_seek(self, video_path: str | Path, start: int, end: int) -> List[torch.Tensor]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start))

        frames = []
        for _ in range(int(start), int(end)):
            ok, frame = cap.read()
            if not ok:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform is not None:
                rgb = self.transform(rgb)
            frames.append(rgb)

        cap.release()
        return frames

    def _decode_frames_sequential(self, video_path: str | Path, start: int, end: int) -> List[torch.Tensor]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        frames = []
        idx = 0

        while idx < int(end):
            ok, frame = cap.read()
            if not ok:
                break

            if idx >= int(start):
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transform is not None:
                    rgb = self.transform(rgb)
                frames.append(rgb)

            idx += 1

        cap.release()
        return frames

    def _read_window_frames(self, video_path: str | Path, start: int, end: int) -> torch.Tensor:
        expected = int(end) - int(start)

        frames = self._decode_frames_from_seek(video_path, start, end)

        if len(frames) != expected and self.cfg.sequential_read_fallback:
            frames = self._decode_frames_sequential(video_path, start, end)

        if len(frames) != expected:
            raise RuntimeError(
                f"Frame read mismatch after fallback: {video_path}, start={start}, end={end}, "
                f"expected={expected}, got={len(frames)}. "
                f"Try --verify-decoded-frame-count and rerun indexing."
            )

        return torch.stack(frames, dim=0)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        s = self.samples[idx]
        labels = load_label_csv(s["label_path"])[s["start"]:s["end"]]
        labels_t = torch.tensor(labels, dtype=torch.float32)

        frames = self._read_window_frames(s["video_path"], s["start"], s["end"])

        return {
            "video_id": s["video_id"],
            "start": s["start"],
            "end": s["end"],
            "window_has_fight": s["window_has_fight"],
            "frames": frames,
            "labels": labels_t,
        }


def make_balanced_window_sampler(dataset: NTUCCTVWindowDataset) -> Optional[WeightedRandomSampler]:
    if len(dataset) == 0:
        return None

    window_labels = np.array([s["window_has_fight"] for s in dataset.samples], dtype=np.int64)
    class_counts = np.bincount(window_labels, minlength=2)

    if class_counts.min() == 0:
        print("[WARN] Only one window class exists. Using normal shuffle.")
        return None

    class_weights = 1.0 / class_counts
    sample_weights = class_weights[window_labels]

    print("Window class counts [no-fight, fight]:", class_counts.tolist())
    print("Window class weights:", class_weights.tolist())

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, cfg: Config) -> Tuple[DataLoader, DataLoader]:
    image_transform = make_image_transform(cfg)

    train_dataset = NTUCCTVWindowDataset(
        train_df,
        cfg=cfg,
        window_size=cfg.window_size,
        stride=cfg.train_stride,
        transform=image_transform,
        max_windows_per_video=cfg.max_train_windows_per_video,
        max_total_windows=cfg.max_total_train_windows,
        split_name="train",
        rng_seed=cfg.seed,
    )

    val_dataset = NTUCCTVWindowDataset(
        val_df,
        cfg=cfg,
        window_size=cfg.window_size,
        stride=cfg.train_stride,
        transform=image_transform,
        max_windows_per_video=cfg.max_val_windows_per_video,
        max_total_windows=cfg.max_total_val_windows,
        split_name="val",
        rng_seed=cfg.seed + 1,
    )

    if len(train_dataset) == 0:
        raise RuntimeError("Training dataset has zero windows.")
    if len(val_dataset) == 0:
        raise RuntimeError("Validation dataset has zero windows.")

    sampler = make_balanced_window_sampler(train_dataset) if cfg.use_balanced_window_sampler else None

    loader_kwargs = dict(
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    if cfg.num_workers > 0:
        loader_kwargs.update(persistent_workers=True, prefetch_factor=2)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=(sampler is None),
        sampler=sampler,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        **loader_kwargs,
    )

    batch = next(iter(train_loader))
    print("frames:", batch["frames"].shape)
    print("labels:", batch["labels"].shape)
    print("window_has_fight:", batch["window_has_fight"])

    return train_loader, val_loader


In [ ]:
# ----------------------------
# Model and training
# ----------------------------

class Stage1MobileNetGRU(nn.Module):
    def __init__(
        self,
        hidden_dim: int = 256,
        num_layers: int = 1,
        dropout: float = 0.5,
        freeze_backbone: bool = True,
        pretrained_backbone: bool = True,
    ):
        super().__init__()

        models, _ = require_torchvision()
        try:
            weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained_backbone else None
            backbone = models.mobilenet_v2(weights=weights)
        except AttributeError:
            backbone = models.mobilenet_v2(pretrained=pretrained_backbone)

        self.feature_extractor = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 1280
        self.backbone_frozen = freeze_backbone

        if freeze_backbone:
            for p in self.feature_extractor.parameters():
                p.requires_grad = False

        self.gru = nn.GRU(
            input_size=self.feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)

        feats = self.feature_extractor(x)
        feats = self.pool(feats).flatten(1)
        feats = feats.reshape(B, T, self.feature_dim)

        gru_out, _ = self.gru(feats)
        logits = self.classifier(self.dropout(gru_out)).squeeze(-1)
        return logits


def make_model(cfg: Config) -> Stage1MobileNetGRU:
    model = Stage1MobileNetGRU(
        dropout=cfg.dropout,
        freeze_backbone=cfg.freeze_backbone,
        pretrained_backbone=cfg.pretrained_backbone,
    ).to(cfg.device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(model)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    return model


def keep_frozen_backbone_in_eval_mode(model: Stage1MobileNetGRU) -> None:
    if getattr(model, "backbone_frozen", False):
        model.feature_extractor.eval()


def init_binary_stats() -> Dict[str, int]:
    return {"tp": 0, "fp": 0, "fn": 0, "tn": 0}


def update_binary_stats(stats: Dict[str, int], logits: torch.Tensor, labels: torch.Tensor, threshold: float = 0.5) -> None:
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).long()
    labels = labels.long()

    stats["tp"] += ((preds == 1) & (labels == 1)).sum().item()
    stats["fp"] += ((preds == 1) & (labels == 0)).sum().item()
    stats["fn"] += ((preds == 0) & (labels == 1)).sum().item()
    stats["tn"] += ((preds == 0) & (labels == 0)).sum().item()


def finalize_binary_stats(stats: Dict[str, int]) -> Dict[str, float]:
    tp, fp, fn, tn = stats["tp"], stats["fp"], stats["fn"], stats["tn"]

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + fp + fn + tn, 1)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "acc": acc,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def train_one_epoch(
    model: Stage1MobileNetGRU,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    cfg: Config,
    threshold: float = 0.5,
) -> Dict[str, float]:
    model.train()
    keep_frozen_backbone_in_eval_mode(model)

    running_loss = 0.0
    stats = init_binary_stats()

    print(f"[train] num batches = {len(loader)}")

    for batch_idx, batch in enumerate(loader):
        frames = batch["frames"].to(cfg.device, non_blocking=True)
        labels = batch["labels"].to(cfg.device, non_blocking=True)

        if batch_idx == 0:
            print("[train] frames shape:", frames.shape)
            print("[train] labels shape:", labels.shape)

        optimizer.zero_grad(set_to_none=True)
        logits = model(frames)
        loss = criterion(logits, labels)
        loss.backward()

        if cfg.grad_clip_norm is not None:
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)

        optimizer.step()

        running_loss += loss.item() * frames.size(0)
        update_binary_stats(stats, logits.detach(), labels.detach(), threshold=threshold)

        if batch_idx % 100 == 0:
            partial = finalize_binary_stats(stats)
            avg_so_far = running_loss / max((batch_idx + 1) * frames.size(0), 1)
            print(
                f"[train] batch {batch_idx}/{len(loader)} "
                f"loss={loss.item():.4f} avg_loss={avg_so_far:.4f} "
                f"P={partial['precision']:.4f} R={partial['recall']:.4f} F1={partial['f1']:.4f}"
            )

    avg_loss = running_loss / max(len(loader.dataset), 1)
    metrics = finalize_binary_stats(stats)
    metrics["loss"] = avg_loss
    return metrics


@torch.no_grad()
def evaluate(
    model: Stage1MobileNetGRU,
    loader: DataLoader,
    criterion: nn.Module,
    cfg: Config,
    threshold: float = 0.5,
    name: str = "val",
) -> Dict[str, float]:
    model.eval()

    running_loss = 0.0
    stats = init_binary_stats()

    print(f"[{name}] num batches = {len(loader)}")

    for batch_idx, batch in enumerate(loader):
        frames = batch["frames"].to(cfg.device, non_blocking=True)
        labels = batch["labels"].to(cfg.device, non_blocking=True)

        logits = model(frames)
        loss = criterion(logits, labels)

        running_loss += loss.item() * frames.size(0)
        update_binary_stats(stats, logits, labels, threshold=threshold)

        if batch_idx % 100 == 0:
            partial = finalize_binary_stats(stats)
            avg_so_far = running_loss / max((batch_idx + 1) * frames.size(0), 1)
            print(
                f"[{name}] batch {batch_idx}/{len(loader)} "
                f"loss={loss.item():.4f} avg_loss={avg_so_far:.4f} "
                f"P={partial['precision']:.4f} R={partial['recall']:.4f} F1={partial['f1']:.4f}"
            )

    avg_loss = running_loss / max(len(loader.dataset), 1)
    metrics = finalize_binary_stats(stats)
    metrics["loss"] = avg_loss
    return metrics


def build_checkpoint(
    model: Stage1MobileNetGRU,
    optimizer: torch.optim.Optimizer,
    cfg: Config,
    epoch: int,
    row: Dict[str, Any],
    best_val_f1: Optional[float] = None,
) -> Dict[str, Any]:
    ckpt: Dict[str, Any] = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "config": {
            "img_size": cfg.img_size,
            "window_size": cfg.window_size,
            "train_stride": cfg.train_stride,
            "infer_stride": cfg.infer_stride,
            "batch_size": cfg.batch_size,
            "lr": cfg.lr,
            "weight_decay": cfg.weight_decay,
            "dropout": cfg.dropout,
            "freeze_backbone": cfg.freeze_backbone,
            "pretrained_backbone": cfg.pretrained_backbone,
            "use_balanced_window_sampler": cfg.use_balanced_window_sampler,
            "max_pos_weight": cfg.max_pos_weight,
            "label_length_source": cfg.label_length_source,
            "label_fps_source": cfg.label_fps_source,
            "min_valid_fps": cfg.min_valid_fps,
            "max_valid_fps": cfg.max_valid_fps,
        },
        "metrics": row,
    }

    if best_val_f1 is not None:
        ckpt["best_val_f1"] = best_val_f1

    if cfg.save_optimizer_state:
        ckpt["optimizer_state_dict"] = optimizer.state_dict()

    return ckpt


def plot_history(history_df: pd.DataFrame, paths: RuntimePaths) -> None:
    if len(history_df) == 0:
        return

    fig_path = paths.plot_dir / "training_vs_validation_f1.png"
    plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_f1"], label="train_f1")
    plt.plot(history_df["epoch"], history_df["val_f1"], label="val_f1")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.title("Training vs validation F1")
    plt.legend()
    plt.grid(True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)

    fig_path = paths.plot_dir / "training_vs_validation_loss.png"
    plt.figure(figsize=(8, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs validation loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)


@torch.no_grad()
def collect_loader_probs_and_labels(model: Stage1MobileNetGRU, loader: DataLoader, cfg: Config) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_probs = []
    all_labels = []

    for batch in loader:
        frames = batch["frames"].to(cfg.device, non_blocking=True)
        labels = batch["labels"].cpu().numpy().reshape(-1)

        logits = model(frames)
        probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)

        all_probs.append(probs)
        all_labels.append(labels)

    return np.concatenate(all_probs), np.concatenate(all_labels).astype(np.int32)


def metrics_at_threshold_np(labels: np.ndarray, probs: np.ndarray, threshold: float) -> Dict[str, Any]:
    preds = (probs >= threshold).astype(np.int32)

    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + fp + fn + tn, 1)

    return {
        "threshold": float(threshold),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "acc": acc,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "positive_pred_ratio": float(preds.mean()),
    }


def run_validation_threshold_sweep(
    model: Stage1MobileNetGRU,
    val_loader: DataLoader,
    cfg: Config,
    paths: RuntimePaths,
) -> float:
    val_probs, val_labels_flat = collect_loader_probs_and_labels(model, val_loader, cfg)

    thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
    sweep_rows = [metrics_at_threshold_np(val_labels_flat, val_probs, float(t)) for t in thresholds]
    threshold_sweep_df = pd.DataFrame(sweep_rows)

    sweep_path = paths.pred_dir / "validation_threshold_sweep.csv"
    threshold_sweep_df.to_csv(sweep_path, index=False)
    print("Saved:", sweep_path)

    print(threshold_sweep_df.sort_values("f1", ascending=False).head(10).to_string(index=False))

    best_row = threshold_sweep_df.loc[threshold_sweep_df["f1"].idxmax()]
    best_val_threshold = float(best_row["threshold"])

    print("Best validation threshold:", best_val_threshold)
    print("Best validation F1:", float(best_row["f1"]))
    print("Positive prediction ratio at best threshold:", float(best_row["positive_pred_ratio"]))

    fig_path = paths.plot_dir / "validation_threshold_sweep.png"
    plt.figure(figsize=(8, 4))
    plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["precision"], label="precision")
    plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["recall"], label="recall")
    plt.plot(threshold_sweep_df["threshold"], threshold_sweep_df["f1"], label="f1")
    plt.xlabel("Threshold")
    plt.ylabel("Metric")
    plt.title("Validation threshold sweep")
    plt.legend()
    plt.grid(True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)

    return best_val_threshold


In [ ]:
# ----------------------------
# Full-video inference and post-processing
# ----------------------------

def moving_average(x: Sequence[float], k: int = 5) -> np.ndarray:
    x_np = np.asarray(x, dtype=np.float32)
    if k <= 1:
        return x_np.copy()

    pad = k // 2
    x_pad = np.pad(x_np, (pad, pad), mode="edge")
    kernel = np.ones(k, dtype=np.float32) / k
    y = np.convolve(x_pad, kernel, mode="valid")
    return y[:len(x_np)]


def hysteresis_threshold(probs: Sequence[float], high: float = 0.5, low: float = 0.3) -> np.ndarray:
    preds = np.zeros(len(probs), dtype=np.int32)
    active = False

    for i, p in enumerate(probs):
        if not active and p >= high:
            active = True
        elif active and p < low:
            active = False
        preds[i] = int(active)

    return preds


def fill_short_gaps(preds: Sequence[int], max_gap: int = 5) -> np.ndarray:
    preds_np = np.asarray(preds).astype(np.int32).copy()
    n = len(preds_np)
    i = 0

    while i < n:
        if preds_np[i] == 0:
            j = i
            while j < n and preds_np[j] == 0:
                j += 1

            gap_len = j - i
            has_left = i > 0 and preds_np[i - 1] == 1
            has_right = j < n and preds_np[j] == 1

            if has_left and has_right and gap_len <= max_gap:
                preds_np[i:j] = 1

            i = j
        else:
            i += 1

    return preds_np


def remove_short_segments(preds: Sequence[int], min_len: int = 8) -> np.ndarray:
    preds_np = np.asarray(preds).astype(np.int32).copy()
    n = len(preds_np)
    i = 0

    while i < n:
        if preds_np[i] == 1:
            j = i
            while j < n and preds_np[j] == 1:
                j += 1

            if j - i < min_len:
                preds_np[i:j] = 0

            i = j
        else:
            i += 1

    return preds_np


def pad_positive_segments(preds: Sequence[int], pre_pad: int = 5, post_pad: int = 5) -> np.ndarray:
    preds_np = np.asarray(preds).astype(np.int32)
    out = preds_np.copy()
    n = len(preds_np)
    i = 0

    while i < n:
        if preds_np[i] == 1:
            j = i
            while j < n and preds_np[j] == 1:
                j += 1

            out[max(0, i - pre_pad):min(n, j + post_pad)] = 1
            i = j
        else:
            i += 1

    return out


def postprocess_fight_probs_hysteresis(probs: Sequence[float], cfg: Config) -> Tuple[np.ndarray, np.ndarray]:
    smooth = moving_average(probs, k=cfg.post_smooth_k)
    preds = hysteresis_threshold(smooth, high=cfg.post_high_thresh, low=cfg.post_low_thresh)
    preds = fill_short_gaps(preds, max_gap=cfg.post_max_gap)
    preds = remove_short_segments(preds, min_len=cfg.post_min_len)
    preds = pad_positive_segments(preds, pre_pad=cfg.post_pre_pad, post_pad=cfg.post_post_pad)
    return smooth, preds


def postprocess_probs_at_threshold(probs: Sequence[float], threshold: float, cfg: Config) -> Tuple[np.ndarray, np.ndarray]:
    sm = moving_average(probs, cfg.post_smooth_k)
    pred = (sm >= threshold).astype(np.int32)
    pred = fill_short_gaps(pred, cfg.post_max_gap)
    pred = remove_short_segments(pred, cfg.post_min_len)
    pred = pad_positive_segments(pred, cfg.post_pre_pad, cfg.post_post_pad)
    return sm, pred


def compute_numpy_metrics(labels: Sequence[int], preds: Sequence[int], probs: Optional[Sequence[float]] = None) -> Dict[str, Any]:
    labels_np = np.asarray(labels).astype(int)
    preds_np = np.asarray(preds).astype(int)

    tp = int(((preds_np == 1) & (labels_np == 1)).sum())
    fp = int(((preds_np == 1) & (labels_np == 0)).sum())
    fn = int(((preds_np == 0) & (labels_np == 1)).sum())
    tn = int(((preds_np == 0) & (labels_np == 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(len(labels_np), 1)

    out: Dict[str, Any] = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "acc": acc,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

    if probs is not None and len(np.unique(labels_np)) > 1:
        try:
            out["auc"] = float(roc_auc_score(labels_np, probs))
        except Exception:
            out["auc"] = None
    else:
        out["auc"] = None

    return out


def make_valid_start_positions(num_frames: int, window_size: int, stride: int) -> List[int]:
    num_frames = int(num_frames)
    window_size = int(window_size)
    stride = int(stride)

    if num_frames <= 0:
        raise ValueError("num_frames must be positive")

    if num_frames <= window_size:
        return [0]

    last_start = num_frames - window_size
    starts = list(range(0, last_start + 1, stride))

    if starts[-1] != last_start:
        starts.append(last_start)

    return starts


def read_tensor_window(
    video_path: str | Path,
    start: int,
    end: int,
    image_transform: Any,
    cfg: Config,
    pad_to_expected: bool = True,
) -> torch.Tensor:
    expected = int(end) - int(start)

    if expected <= 0:
        raise ValueError(f"Invalid window: start={start}, end={end}")

    def _read_with_seek() -> List[torch.Tensor]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start))

        frames: List[torch.Tensor] = []
        for _ in range(int(start), int(end)):
            ok, frame = cap.read()
            if not ok:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(image_transform(rgb))

        cap.release()
        return frames

    def _read_sequential() -> List[torch.Tensor]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        frames: List[torch.Tensor] = []
        idx = 0

        while idx < int(end):
            ok, frame = cap.read()
            if not ok:
                break

            if idx >= int(start):
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(image_transform(rgb))

            idx += 1

        cap.release()
        return frames

    frames = _read_with_seek()

    if len(frames) != expected and cfg.sequential_read_fallback:
        frames = _read_sequential()

    if len(frames) == 0:
        raise RuntimeError(
            f"No frames read from {video_path}, start={start}, end={end}. "
            "This usually means label length is longer than the readable video, "
            "or OpenCV cannot decode this video. Check label_len/json_nb_frames/video_frame_count."
        )

    if pad_to_expected:
        while len(frames) < expected:
            frames.append(frames[-1].clone())

    if len(frames) != expected:
        raise RuntimeError(f"Expected {expected} frames, got {len(frames)} for {video_path}")

    return torch.stack(frames)


@torch.no_grad()
def run_full_video_inference(
    model: Stage1MobileNetGRU,
    row: pd.Series | Dict[str, Any],
    cfg: Config,
    raw_threshold: float = 0.5,
    apply_postprocess: bool = True,
    use_hysteresis: bool = True,
) -> Dict[str, Any]:
    model.eval()
    image_transform = make_image_transform(cfg)

    video_path = Path(row["video_path"])
    labels = load_label_csv(row["label_path"])
    usable_len = len(labels)

    window_size = min(int(cfg.window_size), usable_len)
    start_positions = make_valid_start_positions(
        num_frames=usable_len,
        window_size=window_size,
        stride=cfg.infer_stride,
    )

    print("usable_len:", usable_len)
    print("window_size:", window_size)
    print("first starts:", start_positions[:5])
    print("last starts:", start_positions[-5:])
    print("last valid start should be:", max(0, usable_len - window_size))

    prob_sum = np.zeros(usable_len, dtype=np.float32)
    prob_count = np.zeros(usable_len, dtype=np.float32)

    for start in tqdm(start_positions, desc=f"Inference {Path(video_path).stem}"):
        end = min(start + window_size, usable_len)
        clip = read_tensor_window(video_path, start, end, image_transform, cfg)

        if clip.shape[0] < cfg.window_size:
            pad_count = cfg.window_size - clip.shape[0]
            pad_frames = clip[-1:].repeat(pad_count, 1, 1, 1)
            clip = torch.cat([clip, pad_frames], dim=0)

        clip = clip[:cfg.window_size].unsqueeze(0).to(cfg.device)
        logits = model(clip)
        probs = torch.sigmoid(logits).squeeze(0).detach().cpu().numpy()

        real_len = end - start
        prob_sum[start:end] += probs[:real_len]
        prob_count[start:end] += 1.0

    prob_count[prob_count == 0] = 1.0
    raw_probs = prob_sum / prob_count

    raw_preds = (raw_probs >= raw_threshold).astype(np.int32)

    if apply_postprocess:
        if use_hysteresis:
            smoothed_probs, post_preds = postprocess_fight_probs_hysteresis(raw_probs, cfg)
        else:
            smoothed_probs, post_preds = postprocess_probs_at_threshold(raw_probs, raw_threshold, cfg)
    else:
        smoothed_probs, post_preds = raw_probs.copy(), raw_preds.copy()

    return {
        "labels": labels,
        "raw_probs": raw_probs,
        "raw_preds": raw_preds,
        "smoothed_probs": smoothed_probs,
        "post_preds": post_preds,
        "raw_segments": get_positive_ranges(raw_preds),
        "post_segments": get_positive_ranges(post_preds),
    }


def plot_single_full_video(
    video_id: str,
    labels: np.ndarray,
    raw_probs: np.ndarray,
    smoothed_probs: np.ndarray,
    post_preds: np.ndarray,
    paths: RuntimePaths,
) -> Path:
    fig_path = paths.plot_dir / f"{video_id}_full_video_inference.png"
    plt.figure(figsize=(16, 5))
    plt.plot(raw_probs, label="Raw fight probability", alpha=0.7)
    plt.plot(smoothed_probs, label="Smoothed fight probability", alpha=0.9)
    plt.plot(labels, label="Ground truth label", alpha=0.8)
    plt.plot(post_preds, label="Post-processed prediction", alpha=0.8)
    plt.xlabel("Frame index")
    plt.ylabel("Probability / Label")
    plt.title(f"Full-video inference: {video_id}")
    plt.legend()
    plt.grid(True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)
    return fig_path


def evaluate_full_videos(
    model: Stage1MobileNetGRU,
    meta_df: pd.DataFrame,
    cfg: Config,
    max_videos: Optional[int] = 10,
    threshold: Optional[float] = None,
) -> pd.DataFrame:
    rows = []
    selected_df = meta_df.copy()

    if max_videos is not None:
        selected_df = selected_df.head(max_videos)

    all_labels = []
    all_raw_preds = []
    all_post_preds = []
    all_raw_probs = []
    all_smooth_probs = []

    inference_threshold = cfg.train_eval_threshold if threshold is None else threshold
    print("Using inference threshold:", inference_threshold)

    for _, row in selected_df.iterrows():
        video_id = row["video_id"]
        print("\nEvaluating:", video_id)

        result = run_full_video_inference(
            model,
            row,
            cfg,
            raw_threshold=inference_threshold,
            apply_postprocess=True,
            use_hysteresis=True,
        )

        labels = result["labels"]
        raw_preds = result["raw_preds"]
        post_preds = result["post_preds"]
        raw_probs = result["raw_probs"]
        smooth_probs = result["smoothed_probs"]

        raw_m = compute_numpy_metrics(labels, raw_preds, raw_probs)
        post_m = compute_numpy_metrics(labels, post_preds, smooth_probs)

        rows.append({
            "video_id": video_id,
            "num_frames": len(labels),
            "raw_f1": raw_m["f1"],
            "raw_precision": raw_m["precision"],
            "raw_recall": raw_m["recall"],
            "raw_acc": raw_m["acc"],
            "raw_auc": raw_m["auc"],
            "post_f1": post_m["f1"],
            "post_precision": post_m["precision"],
            "post_recall": post_m["recall"],
            "post_acc": post_m["acc"],
            "post_auc": post_m["auc"],
            "gt_segments": get_positive_ranges(labels),
            "post_segments": result["post_segments"],
        })

        all_labels.append(labels)
        all_raw_preds.append(raw_preds)
        all_post_preds.append(post_preds)
        all_raw_probs.append(raw_probs)
        all_smooth_probs.append(smooth_probs)

    full_df = pd.DataFrame(rows)

    if len(all_labels) > 0:
        y = np.concatenate(all_labels)
        rp = np.concatenate(all_raw_preds)
        pp = np.concatenate(all_post_preds)
        rprob = np.concatenate(all_raw_probs)
        sprob = np.concatenate(all_smooth_probs)

        print("\nAggregate raw metrics:")
        print(json.dumps(compute_numpy_metrics(y, rp, rprob), indent=2))

        print("\nAggregate post-processed metrics:")
        print(json.dumps(compute_numpy_metrics(y, pp, sprob), indent=2))

    return full_df


In [ ]:
# ----------------------------
# Grad-CAM
# ----------------------------

def read_raw_and_tensor_window(
    video_path: str | Path,
    start: int,
    end: int,
    image_transform: Any,
    cfg: Config,
) -> Tuple[List[np.ndarray], torch.Tensor]:
    expected = int(end) - int(start)

    def _read_with_seek() -> Tuple[List[np.ndarray], List[torch.Tensor]]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start))

        raw_frames: List[np.ndarray] = []
        tensor_frames: List[torch.Tensor] = []

        for _ in range(int(start), int(end)):
            ok, frame = cap.read()
            if not ok:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            raw_display = cv2.resize(rgb, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_AREA)
            raw_frames.append(raw_display)
            tensor_frames.append(image_transform(rgb))

        cap.release()
        return raw_frames, tensor_frames

    def _read_sequential() -> Tuple[List[np.ndarray], List[torch.Tensor]]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {video_path}")

        raw_frames: List[np.ndarray] = []
        tensor_frames: List[torch.Tensor] = []
        idx = 0

        while idx < int(end):
            ok, frame = cap.read()
            if not ok:
                break

            if idx >= int(start):
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                raw_display = cv2.resize(rgb, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_AREA)
                raw_frames.append(raw_display)
                tensor_frames.append(image_transform(rgb))

            idx += 1

        cap.release()
        return raw_frames, tensor_frames

    raw_frames, tensor_frames = _read_with_seek()

    if len(tensor_frames) != expected and cfg.sequential_read_fallback:
        raw_frames, tensor_frames = _read_sequential()

    if len(tensor_frames) != expected:
        raise RuntimeError(f"Expected {expected} frames, got {len(tensor_frames)}")

    return raw_frames, torch.stack(tensor_frames, dim=0)


class TemporalGradCAM:
    def __init__(self, model: Stage1MobileNetGRU, target_layer: nn.Module, device: str):
        self.model = model
        self.target_layer = target_layer
        self.device = device
        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(self._save_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove(self) -> None:
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, clip_tensor: torch.Tensor, target_frame_idx: Optional[int] = None) -> Dict[str, Any]:
        was_training = self.model.training
        self.model.eval()
        self.model.zero_grad(set_to_none=True)

        clip_tensor = clip_tensor.detach().clone().to(self.device)
        clip_tensor.requires_grad_(True)

        with torch.backends.cudnn.flags(enabled=False):
            with torch.enable_grad():
                logits = self.model(clip_tensor)
                probs = torch.sigmoid(logits)

                if target_frame_idx is None:
                    target_frame_idx = int(torch.argmax(probs[0]).item())

                score = logits[0, target_frame_idx]
                score.backward()

        if self.activations is None or self.gradients is None:
            raise RuntimeError("Grad-CAM did not capture activations/gradients.")

        acts = self.activations.detach()[target_frame_idx]
        grads = self.gradients.detach()[target_frame_idx]

        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = (weights * acts).sum(dim=0)
        cam = torch.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        if was_training:
            self.model.train()

        return {
            "cam": cam.cpu().numpy(),
            "target_frame_idx": int(target_frame_idx),
            "target_prob": float(probs[0, target_frame_idx].detach().cpu().item()),
        }


def make_gradcam_overlay(raw_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> Tuple[np.ndarray, np.ndarray]:
    h, w = raw_rgb.shape[:2]
    cam_resized = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)
    heat = np.uint8(255 * cam_resized)

    heat_bgr = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
    heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(raw_rgb, 1.0 - alpha, heat_rgb, alpha, 0)
    return heat_rgb, overlay


def save_gradcam_figure(raw_rgb: np.ndarray, heat_rgb: np.ndarray, overlay_rgb: np.ndarray, save_path: Path, title: str = "Grad-CAM") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(raw_rgb)
    axes[0].set_title("Input frame")
    axes[0].axis("off")

    axes[1].imshow(heat_rgb)
    axes[1].set_title("Grad-CAM heatmap")
    axes[1].axis("off")

    axes[2].imshow(overlay_rgb)
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("Saved Grad-CAM figure:", save_path)


def run_gradcam_for_sample(
    model: Stage1MobileNetGRU,
    sample_row: pd.Series,
    result: Dict[str, Any],
    cfg: Config,
    paths: RuntimePaths,
) -> Optional[Path]:
    labels = result["labels"]
    probs = result["smoothed_probs"]

    fight_idxs = np.where(labels == 1)[0]
    if len(fight_idxs) > 0:
        center_frame = int(fight_idxs[np.argmax(probs[fight_idxs])])
    else:
        center_frame = int(np.argmax(probs))

    start = int(np.clip(center_frame - cfg.window_size // 2, 0, max(0, len(labels) - cfg.window_size)))
    end = start + cfg.window_size
    target_frame_idx = center_frame - start

    sample_video_id = sample_row["video_id"]
    sample_video_path = Path(sample_row["video_path"])

    print("Grad-CAM video:", sample_video_id)
    print("Global frame:", center_frame)
    print("Window:", start, "to", end)
    print("Local frame:", target_frame_idx)

    image_transform = make_image_transform(cfg)
    raw_frames, clip_tensor = read_raw_and_tensor_window(sample_video_path, start, end, image_transform, cfg)
    clip_tensor = clip_tensor.unsqueeze(0)

    gradcam = TemporalGradCAM(model, model.feature_extractor[-1], cfg.device)
    try:
        cam_result = gradcam.generate(clip_tensor, target_frame_idx=target_frame_idx)
    finally:
        gradcam.remove()

    cam = cam_result["cam"]
    target_prob = cam_result["target_prob"]

    raw_rgb = raw_frames[target_frame_idx]
    heat_rgb, overlay_rgb = make_gradcam_overlay(raw_rgb, cam)

    save_path = paths.gradcam_dir / f"{sample_video_id}_frame_{center_frame:06d}_gradcam.png"
    title = f"Grad-CAM | {sample_video_id} | frame={center_frame} | prob={target_prob:.3f}"
    save_gradcam_figure(raw_rgb, heat_rgb, overlay_rgb, save_path, title=title)
    return save_path


In [ ]:
# ----------------------------
# Checkpoints
# ----------------------------

def torch_load(path: Path, device: str) -> Any:
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def checkpoint_candidates(cfg: Config) -> List[Path]:
    candidates = []
    if cfg.checkpoint is not None:
        candidates.append(Path(cfg.checkpoint))
    candidates.extend([
        cfg.base_dir / "ntu_cctv_stage1_outputs_all_folders_fixed_labels/checkpoints/stage1_best_ntu_all_folders.pt",
        cfg.base_dir / "ntu_cctv_stage1_outputs_all_folders_fixed_labels/checkpoints/stage1_best_ntu_001_200.pt",
        cfg.base_dir / "ntu_cctv_stage1_outputs_001_200_fixed_labels/checkpoints/stage1_best_ntu_001_200.pt",
        cfg.output_dir / "checkpoints/stage1_best_ntu_all_folders.pt",
    ])
    return candidates


def find_checkpoint_path(cfg: Config, required: bool = True) -> Optional[Path]:
    for p in checkpoint_candidates(cfg):
        p = Path(p)
        if p.exists():
            return p
    if required:
        raise FileNotFoundError("No checkpoint found. Pass --checkpoint.")
    return None


def load_checkpoint_into_model(model: Stage1MobileNetGRU, ckpt_path: Path, cfg: Config) -> Any:
    print("Loading:", ckpt_path)
    ckpt = torch_load(ckpt_path, cfg.device)
    model.load_state_dict(ckpt.get("model_state_dict", ckpt))
    model.to(cfg.device)
    model.eval()

    print("Loaded checkpoint.")
    if isinstance(ckpt, dict):
        print("epoch:", ckpt.get("epoch"))
        print("best_val_f1:", ckpt.get("best_val_f1"))
        print("metrics:", ckpt.get("metrics"))
    return ckpt


In [ ]:
# ----------------------------
# Main workflows
# ----------------------------

def prepare_index(cfg: Config) -> Tuple[pd.DataFrame, RuntimePaths]:
    paths = make_runtime_paths(cfg)

    print("Device:", cfg.device)
    print("BASE_DIR:", cfg.base_dir)
    print("VIDEO_DIRS:")
    for d in cfg.video_dirs:
        print("  ", d, "| exists:", d.exists())
    print("OUTPUT_DIR:", cfg.output_dir)
    print("VERIFY_DECODED_FRAME_COUNT:", cfg.verify_decoded_frame_count)
    print("SEQUENTIAL_READ_FALLBACK:", cfg.sequential_read_fallback)
    print("MAX_VIDEOS:", cfg.max_videos)

    _, gt = load_groundtruth_json(cfg)
    video_map, _ = build_video_map(cfg.video_dirs)
    index_df = build_index_from_json(gt["database"], video_map, cfg, paths)
    return index_df, paths


def run_train(cfg: Config) -> None:
    index_df, paths = prepare_index(cfg)

    if cfg.export_gt_overlay:
        export_gt_overlay_sample(index_df, cfg, paths)

    train_df, val_df, test_df = make_splits(index_df, cfg)

    print("Train videos:", len(train_df))
    print("Val videos:", len(val_df))
    print("Test videos:", len(test_df))

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        total_frames = int(df["label_len"].sum()) if len(df) else 0
        fight_frames = int(df["fight_frames"].sum()) if len(df) else 0
        print(f"{name}: frames={total_frames:,}, fight={fight_frames:,}, ratio={fight_frames / max(total_frames, 1):.4f}")

    train_loader, val_loader = make_dataloaders(train_df, val_df, cfg)
    model = make_model(cfg)

    train_fight = int(train_df["fight_frames"].sum())
    train_nonfight = int(train_df["nonfight_frames"].sum())
    raw_pos_weight = train_nonfight / max(train_fight, 1)
    pos_weight_value = min(float(raw_pos_weight), float(cfg.max_pos_weight))

    print("train_fight:", train_fight)
    print("train_nonfight:", train_nonfight)
    print("raw pos_weight:", raw_pos_weight)
    print("capped pos_weight:", pos_weight_value)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight_value, device=cfg.device, dtype=torch.float32)
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    history = []
    best_val_f1 = -1.0
    epochs_without_improvement = 0

    best_ckpt_path = paths.checkpoint_dir / "stage1_best_ntu_all_folders.pt"

    for epoch in range(1, cfg.num_epochs + 1):
        print(f"\n=== Epoch {epoch}/{cfg.num_epochs} ===")

        train_m = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            cfg,
            threshold=cfg.train_eval_threshold,
        )

        val_m = evaluate(
            model,
            val_loader,
            criterion,
            cfg,
            threshold=cfg.train_eval_threshold,
            name="val",
        )

        row = {
            "epoch": epoch,
            "train_loss": train_m["loss"],
            "train_precision": train_m["precision"],
            "train_recall": train_m["recall"],
            "train_f1": train_m["f1"],
            "train_acc": train_m["acc"],
            "train_tp": train_m["tp"],
            "train_fp": train_m["fp"],
            "train_fn": train_m["fn"],
            "train_tn": train_m["tn"],
            "val_loss": val_m["loss"],
            "val_precision": val_m["precision"],
            "val_recall": val_m["recall"],
            "val_f1": val_m["f1"],
            "val_acc": val_m["acc"],
            "val_tp": val_m["tp"],
            "val_fp": val_m["fp"],
            "val_fn": val_m["fn"],
            "val_tn": val_m["tn"],
        }

        history.append(row)

        if cfg.save_each_epoch:
            epoch_ckpt_path = paths.epoch_ckpt_dir / f"stage1_epoch_{epoch:03d}.pt"
            torch.save(build_checkpoint(model, optimizer, cfg, epoch=epoch, row=row, best_val_f1=best_val_f1), epoch_ckpt_path)
            print("Saved epoch checkpoint to", epoch_ckpt_path)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={row['train_loss']:.4f} train_P={row['train_precision']:.4f} "
            f"train_R={row['train_recall']:.4f} train_F1={row['train_f1']:.4f} | "
            f"val_loss={row['val_loss']:.4f} val_P={row['val_precision']:.4f} "
            f"val_R={row['val_recall']:.4f} val_F1={row['val_f1']:.4f}"
        )

        if row["val_f1"] > best_val_f1:
            best_val_f1 = row["val_f1"]
            epochs_without_improvement = 0

            torch.save(build_checkpoint(model, optimizer, cfg, epoch=epoch, row=row, best_val_f1=best_val_f1), best_ckpt_path)
            print("Saved best checkpoint to", best_ckpt_path)
        else:
            epochs_without_improvement += 1
            print(f"No val F1 improvement for {epochs_without_improvement}/{cfg.patience} epochs")

        if epochs_without_improvement >= cfg.patience:
            print("Early stopping triggered.")
            break

    history_df = pd.DataFrame(history)
    history_path = cfg.output_dir / "training_history.csv"
    history_df.to_csv(history_path, index=False)
    print("Saved history to", history_path)
    print(history_df.to_string(index=False))
    plot_history(history_df, paths)

    # Learning sanity check.
    if len(history_df) > 0:
        first = history_df.iloc[0]
        last = history_df.iloc[-1]
        best = history_df.loc[history_df["val_f1"].idxmax()]
        print("First epoch train loss:", float(first["train_loss"]))
        print("Last epoch train loss:", float(last["train_loss"]))
        print("First epoch train F1:", float(first["train_f1"]))
        print("Last epoch train F1:", float(last["train_f1"]))
        print("Best val F1:", float(best["val_f1"]), "at epoch", int(best["epoch"]))

        if float(last["train_loss"]) < float(first["train_loss"]) or float(last["train_f1"]) > float(first["train_f1"]):
            print("Conclusion: training is happening on the training split.")
        else:
            print("[WARN] Training metrics did not improve. Check labels, trainable parameters, and LR.")

    best_val_threshold = run_validation_threshold_sweep(model, val_loader, cfg, paths)

    if best_ckpt_path.exists():
        ckpt = torch_load(best_ckpt_path, cfg.device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.to(cfg.device)
        model.eval()
        print("Loaded best checkpoint:", best_ckpt_path)
        print("Best val F1:", ckpt.get("best_val_f1"))

    sample_eval_df = val_df if len(val_df) > 0 else train_df
    sample_row = sample_eval_df.iloc[0]
    sample_video_id = sample_row["video_id"]
    print("Sample video:", sample_video_id)
    print("Using inference threshold:", best_val_threshold)

    result = run_full_video_inference(
        model,
        sample_row,
        cfg,
        raw_threshold=best_val_threshold,
        apply_postprocess=True,
        use_hysteresis=True,
    )

    labels = result["labels"]
    raw_probs = result["raw_probs"]
    raw_preds = result["raw_preds"]
    smoothed_probs = result["smoothed_probs"]
    post_preds = result["post_preds"]

    raw_metrics = compute_numpy_metrics(labels, raw_preds, raw_probs)
    post_metrics = compute_numpy_metrics(labels, post_preds, smoothed_probs)

    print("Raw metrics:")
    print(json.dumps(raw_metrics, indent=2))
    print("\nPost-processed metrics:")
    print(json.dumps(post_metrics, indent=2))
    print("\nRaw segments:", result["raw_segments"][:10])
    print("Post segments:", result["post_segments"][:10])
    print("GT segments:", get_positive_ranges(labels)[:10])

    plot_single_full_video(sample_video_id, labels, raw_probs, smoothed_probs, post_preds, paths)

    output_json = {
        "video_id": sample_video_id,
        "num_frames": int(len(labels)),
        "raw_pred_labels": raw_preds.tolist(),
        "post_pred_labels": post_preds.tolist(),
        "raw_pred_probs": [float(x) for x in raw_probs.tolist()],
        "smoothed_pred_probs": [float(x) for x in smoothed_probs.tolist()],
        "gt_labels": labels.tolist(),
        "raw_metrics": raw_metrics,
        "post_metrics": post_metrics,
        "raw_segments": result["raw_segments"],
        "post_segments": result["post_segments"],
        "gt_segments": get_positive_ranges(labels),
    }

    out_path = paths.pred_dir / f"{sample_video_id}_postprocessed.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output_json, f)
    print("Saved prediction JSON:", out_path)

    if cfg.export_sample_overlay:
        pred_overlay_path = paths.label_video_dir / f"{sample_video_id}_gt_plus_pred.mp4"
        export_labeled_video(
            Path(sample_row["video_path"]),
            labels,
            pred_overlay_path,
            cfg=cfg,
            pred_labels=post_preds,
            pred_probs=smoothed_probs,
            max_frames=cfg.label_video_max_frames,
            resize_width=cfg.label_video_resize_width,
            fps=float(sample_row["label_fps_used"]),
            extra_text="GT + postprocessed prediction",
        )

    if cfg.max_full_eval_videos is not None and cfg.max_full_eval_videos > 0:
        eval_df = evaluate_full_videos(
            model,
            val_df if len(val_df) > 0 else test_df,
            cfg,
            max_videos=cfg.max_full_eval_videos,
            threshold=best_val_threshold,
        )

        eval_path = cfg.output_dir / "full_video_eval.csv"
        eval_df.to_csv(eval_path, index=False)
        print("Saved full-video eval:", eval_path)
        print(eval_df.to_string(index=False))

    if cfg.run_gradcam:
        run_gradcam_for_sample(model, sample_row, result, cfg, paths)


def split_df_by_name(index_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if split_name == "all":
        return index_df.copy()
    return index_df[index_df["subset"].str.lower() == split_name.lower()].reset_index(drop=True)


def eval_thresholds(labels: np.ndarray, raw_probs: np.ndarray, thresholds: Sequence[float], cfg: Config) -> Tuple[pd.DataFrame, Dict[float, Dict[str, np.ndarray]]]:
    rows, pred_dict = [], {}

    for thr in thresholds:
        thr = float(thr)
        raw_pred = (raw_probs >= thr).astype(np.int32)
        sm, post_pred = postprocess_probs_at_threshold(raw_probs, thr, cfg)

        raw_m = compute_numpy_metrics(labels, raw_pred)
        post_m = compute_numpy_metrics(labels, post_pred)

        rows.append({
            "threshold": thr,
            "raw_precision": raw_m["precision"],
            "raw_recall": raw_m["recall"],
            "raw_f1": raw_m["f1"],
            "raw_acc": raw_m["acc"],
            "raw_auc": raw_m["auc"],
            "post_precision": post_m["precision"],
            "post_recall": post_m["recall"],
            "post_f1": post_m["f1"],
            "post_acc": post_m["acc"],
            "post_auc": post_m["auc"],
            "positive_ratio_post": float(post_pred.mean()),
            "post_segments": get_positive_ranges(post_pred),
        })

        pred_dict[thr] = {"smoothed_probs": sm, "post_pred": post_pred, "raw_pred": raw_pred}

    return pd.DataFrame(rows), pred_dict


def plot_threshold_sweep(thr_df: pd.DataFrame, video_id: str, paths: RuntimePaths) -> Path:
    fig_path = paths.plot_dir / f"{video_id}_threshold_sweep.png"
    plt.figure(figsize=(8, 5))
    plt.plot(thr_df["threshold"], thr_df["post_precision"], marker="o", label="precision")
    plt.plot(thr_df["threshold"], thr_df["post_recall"], marker="o", label="recall")
    plt.plot(thr_df["threshold"], thr_df["post_f1"], marker="o", label="F1")
    plt.plot(thr_df["threshold"], thr_df["positive_ratio_post"], marker="o", label="positive ratio")
    plt.xlabel("Threshold")
    plt.ylabel("Metric")
    plt.title(f"Threshold sweep: {video_id}")
    plt.grid(True)
    plt.legend()
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)
    return fig_path


def plot_multi_threshold(
    video_id: str,
    labels: np.ndarray,
    raw_probs: np.ndarray,
    preds_by_thr: Dict[float, Dict[str, np.ndarray]],
    thresholds: Sequence[float],
    paths: RuntimePaths,
) -> Path:
    x = np.arange(len(labels))
    fig_path = paths.plot_dir / f"{video_id}_multi_threshold_plot.png"

    plt.figure(figsize=(16, 7))
    plt.plot(x, raw_probs, label="Raw fight probability", alpha=0.6)
    plt.plot(x, preds_by_thr[float(thresholds[0])]["smoothed_probs"], label="Smoothed probability", linewidth=2)
    plt.step(x, labels, where="post", label="Ground truth", linewidth=2)

    offset = 0.08
    for i, thr in enumerate(thresholds):
        pred = preds_by_thr[float(thr)]["post_pred"]
        plt.step(x, pred.astype(float) - (i + 1) * offset, where="post", label=f"Post pred @ {thr:.2f}")

    plt.ylim(-offset * (len(thresholds) + 1), 1.05)
    plt.xlabel("Frame index")
    plt.ylabel("Probability / Label")
    plt.title(f"Full-video inference with multiple thresholds: {video_id}")
    plt.grid(True)
    plt.legend(loc="upper left")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()

    print("Saved:", fig_path)
    return fig_path


@torch.no_grad()
def collect_probs_for_posttest(model: Stage1MobileNetGRU, meta_df: pd.DataFrame, cfg: Config, max_videos: Optional[int]) -> pd.DataFrame:
    meta_df = meta_df.head(max_videos) if max_videos is not None else meta_df
    rows = []

    for i, (_, row) in enumerate(meta_df.iterrows(), start=1):
        print(f"[{i}/{len(meta_df)}] {row['video_id']}")
        res = run_full_video_inference(
            model,
            row,
            cfg,
            raw_threshold=cfg.raw_threshold,
            apply_postprocess=True,
            use_hysteresis=False,
        )
        rows.append({
            "video_id": row["video_id"],
            "subset": row["subset"],
            "num_frames": len(res["labels"]),
            "labels": res["labels"],
            "raw_probs": res["raw_probs"],
        })

    return pd.DataFrame(rows)


def aggregate_counts(metric_rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    tp = sum(int(m["tp"]) for m in metric_rows)
    fp = sum(int(m["fp"]) for m in metric_rows)
    fn = sum(int(m["fn"]) for m in metric_rows)
    tn = sum(int(m["tn"]) for m in metric_rows)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    acc = (tp + tn) / max(tp + fp + fn + tn, 1)
    return {"precision": precision, "recall": recall, "f1": f1, "acc": acc, "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def global_threshold_sweep(full_probs_df: pd.DataFrame, cfg: Config, paths: RuntimePaths) -> Tuple[pd.DataFrame, float]:
    global_rows = []
    for thr in cfg.global_thresholds:
        raw_metrics = []
        post_metrics = []
        positive_ratios = []

        for _, row in full_probs_df.iterrows():
            labels = row["labels"]
            probs = row["raw_probs"]

            raw_pred = (probs >= float(thr)).astype(np.int32)
            sm, post_pred = postprocess_probs_at_threshold(probs, float(thr), cfg)

            raw_metrics.append(compute_numpy_metrics(labels, raw_pred))
            post_metrics.append(compute_numpy_metrics(labels, post_pred))
            positive_ratios.append(float(post_pred.mean()))

        raw_m = aggregate_counts(raw_metrics)
        post_m = aggregate_counts(post_metrics)

        global_rows.append({
            "threshold": float(thr),
            "raw_precision": raw_m["precision"],
            "raw_recall": raw_m["recall"],
            "raw_f1": raw_m["f1"],
            "raw_acc": raw_m["acc"],
            "post_precision": post_m["precision"],
            "post_recall": post_m["recall"],
            "post_f1": post_m["f1"],
            "post_acc": post_m["acc"],
            "positive_ratio_post_mean": float(np.mean(positive_ratios)) if positive_ratios else 0.0,
        })

    global_thr_df = pd.DataFrame(global_rows)
    print(global_thr_df.sort_values("post_f1", ascending=False).head(10).to_string(index=False))

    best_global_threshold = float(global_thr_df.loc[global_thr_df["post_f1"].idxmax(), "threshold"])
    print("Best global threshold:", best_global_threshold)

    global_thr_path = paths.pred_dir / "global_threshold_sweep.csv"
    global_thr_df.to_csv(global_thr_path, index=False)
    print("Saved:", global_thr_path)

    fig_path = paths.plot_dir / "global_threshold_sweep.png"
    plt.figure(figsize=(8, 5))
    plt.plot(global_thr_df["threshold"], global_thr_df["post_precision"], marker="o", label="precision")
    plt.plot(global_thr_df["threshold"], global_thr_df["post_recall"], marker="o", label="recall")
    plt.plot(global_thr_df["threshold"], global_thr_df["post_f1"], marker="o", label="F1")
    plt.plot(global_thr_df["threshold"], global_thr_df["positive_ratio_post_mean"], marker="o", label="positive ratio mean")
    plt.xlabel("Threshold")
    plt.ylabel("Metric")
    plt.title(f"Global threshold sweep on {len(full_probs_df)} videos")
    plt.grid(True)
    plt.legend()
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved:", fig_path)

    return global_thr_df, best_global_threshold


def per_video_eval(full_probs_df: pd.DataFrame, threshold: float, cfg: Config) -> pd.DataFrame:
    rows = []

    for _, row in full_probs_df.iterrows():
        labels = row["labels"]
        probs = row["raw_probs"]

        raw_pred = (probs >= threshold).astype(np.int32)
        sm, post_pred = postprocess_probs_at_threshold(probs, threshold, cfg)

        raw_m = compute_numpy_metrics(labels, raw_pred)
        post_m = compute_numpy_metrics(labels, post_pred)

        rows.append({
            "video_id": row["video_id"],
            "subset": row["subset"],
            "num_frames": row["num_frames"],
            "threshold": threshold,
            "raw_precision": raw_m["precision"],
            "raw_recall": raw_m["recall"],
            "raw_f1": raw_m["f1"],
            "raw_acc": raw_m["acc"],
            "raw_auc": raw_m["auc"],
            "post_precision": post_m["precision"],
            "post_recall": post_m["recall"],
            "post_f1": post_m["f1"],
            "post_acc": post_m["acc"],
            "post_auc": post_m["auc"],
            "gt_segments": get_positive_ranges(labels),
            "post_segments": get_positive_ranges(post_pred),
        })

    return pd.DataFrame(rows)


def run_posttest(cfg: Config) -> None:
    index_df, paths = prepare_index(cfg)

    selected_df = split_df_by_name(index_df, cfg.eval_split)
    if len(selected_df) == 0:
        print(f"[WARN] No videos found for split={cfg.eval_split}; using all videos.")
        selected_df = index_df.copy()

    model = make_model(cfg)
    ckpt_path = find_checkpoint_path(cfg, required=True)
    load_checkpoint_into_model(model, ckpt_path, cfg)

    if cfg.sample_video_id is None:
        sample_row = selected_df.iloc[0]
        sample_video_id = sample_row["video_id"]
    else:
        matches = index_df[index_df["video_id"] == cfg.sample_video_id]
        if len(matches) == 0:
            raise ValueError(f"sample_video_id not found in index: {cfg.sample_video_id}")
        sample_row = matches.iloc[0]
        sample_video_id = cfg.sample_video_id

    print("Sample:", sample_video_id)
    sample_labels = load_label_csv(sample_row["label_path"])
    print("label_len:", len(sample_labels))
    print("json_nb_frames:", sample_row["json_nb_frames"])
    print("video_frame_count:", sample_row["video_frame_count"])
    print("label_length_source:", sample_row["label_length_source_used"])
    print("GT ranges:", get_positive_ranges(sample_labels))
    print("GT ranges seconds:", sample_row["segments_sec_from_labels"])

    result = run_full_video_inference(
        model,
        sample_row,
        cfg,
        raw_threshold=cfg.raw_threshold,
        apply_postprocess=True,
        use_hysteresis=False,
    )

    print("Raw:", json.dumps(compute_numpy_metrics(result["labels"], result["raw_preds"]), indent=2))
    print("Post:", json.dumps(compute_numpy_metrics(result["labels"], result["post_preds"]), indent=2))

    thr_df, thr_preds = eval_thresholds(result["labels"], result["raw_probs"], cfg.thresholds_to_plot, cfg)
    thr_path = paths.pred_dir / f"{sample_video_id}_threshold_sweep.csv"
    thr_df.to_csv(thr_path, index=False)
    print("Saved:", thr_path)
    print(thr_df.sort_values("post_f1", ascending=False).to_string(index=False))

    best_video_threshold = float(thr_df.loc[thr_df["post_f1"].idxmax(), "threshold"])
    print("Best threshold for this video:", best_video_threshold)

    plot_threshold_sweep(thr_df, sample_video_id, paths)
    plot_multi_threshold(sample_video_id, result["labels"], result["raw_probs"], thr_preds, cfg.thresholds_to_plot, paths)

    full_probs_df = collect_probs_for_posttest(model, selected_df, cfg, cfg.max_eval_videos)
    print("Collected videos:", len(full_probs_df))
    if len(full_probs_df) > 0:
        print("Collected frames:", int(full_probs_df["num_frames"].sum()))

    _, best_global_threshold = global_threshold_sweep(full_probs_df, cfg, paths)

    eval_df = per_video_eval(full_probs_df, best_global_threshold, cfg)
    eval_path = paths.pred_dir / "per_video_eval_best_global_threshold.csv"
    eval_df.to_csv(eval_path, index=False)
    print("Saved:", eval_path)
    print(eval_df.sort_values("post_f1").head(20).to_string(index=False))
    print(eval_df.describe(numeric_only=True).to_string())

    if cfg.export_sample_overlay:
        sm, post_pred = postprocess_probs_at_threshold(result["raw_probs"], best_global_threshold, cfg)
        overlay_path = paths.label_video_dir / f"{sample_video_id}_gt_pred_thr_{best_global_threshold:.2f}.mp4"
        export_labeled_video(
            Path(sample_row["video_path"]),
            result["labels"],
            overlay_path,
            cfg=cfg,
            pred_labels=post_pred,
            pred_probs=sm,
            max_frames=cfg.label_video_max_frames,
            resize_width=cfg.label_video_resize_width,
            fps=float(sample_row["label_fps_used"]),
            extra_text=f"thr={best_global_threshold:.2f}",
        )


def run_index_only(cfg: Config) -> None:
    index_df, paths = prepare_index(cfg)
    if cfg.export_gt_overlay:
        export_gt_overlay_sample(index_df, cfg, paths)


In [ ]:
import shutil
from IPython.display import Video, display

def copy_ntucctv_data_to_local(
    source_base_dir,
    local_base_dir,
    video_subdirs=DEFAULT_VIDEO_SUBDIRS,
    gt_json_name="groundtruth.json",
    overwrite=False,
):
    source_base_dir = Path(source_base_dir)
    local_base_dir = Path(local_base_dir)
    local_base_dir.mkdir(parents=True, exist_ok=True)

    gt_src = source_base_dir / gt_json_name
    gt_dst = local_base_dir / gt_json_name
    if gt_src.exists():
        if overwrite or not gt_dst.exists():
            shutil.copy2(gt_src, gt_dst)
            print("Copied:", gt_src, "->", gt_dst)
        else:
            print("Already exists, skipped:", gt_dst)
    else:
        print("[WARN] Missing groundtruth.json at:", gt_src)

    for subdir in video_subdirs:
        src_dir = source_base_dir / subdir
        dst_dir = local_base_dir / subdir

        if not src_dir.exists():
            print("[WARN] Missing source video folder:", src_dir)
            continue

        if dst_dir.exists():
            if overwrite:
                shutil.rmtree(dst_dir)
            else:
                print("Already exists, skipped folder:", dst_dir)
                continue

        print("Copying folder:", src_dir, "->", dst_dir)
        shutil.copytree(src_dir, dst_dir)

    print("Local copy complete:", local_base_dir)


def ensure_index_for_notebook(cfg, force_rebuild=False):
    """
    Build/reuse the index_df and paths globals.
    """
    global index_df, paths
    if force_rebuild or "index_df" not in globals() or "paths" not in globals():
        index_df, paths = prepare_index(cfg)
    return index_df, paths


def load_trained_model_for_notebook(cfg, checkpoint_path=None, pretrained_backbone_for_load=False):
    """
    Load a trained checkpoint into Stage1MobileNetGRU.
    pretrained_backbone_for_load=False avoids downloading ImageNet weights during post-testing.
    """
    old_checkpoint = cfg.checkpoint
    old_pretrained = cfg.pretrained_backbone

    if checkpoint_path is not None:
        cfg.checkpoint = Path(checkpoint_path)

    cfg.pretrained_backbone = bool(pretrained_backbone_for_load)
    model = make_model(cfg)
    ckpt_path = find_checkpoint_path(cfg, required=True)
    ckpt = load_checkpoint_into_model(model, ckpt_path, cfg)

    cfg.pretrained_backbone = old_pretrained
    if checkpoint_path is None:
        cfg.checkpoint = old_checkpoint

    return model, ckpt_path, ckpt


def export_single_video_created_labels(
    video_id,
    cfg,
    checkpoint_path=None,
    threshold=None,
    save_dir=None,
    force_rebuild_index=False,
    export_overlay=True,
    show_plot=True,
):
    """
    Single-video workflow from ntu_cctv_sample.ipynb.

    Saves:
    - side-by-side CSV with GT labels, created labels, probabilities, and label_match
    - created-label-only CSV for downstream Stage 2 work
    - summary JSON with GT/predicted segments and metrics
    - optional GT + prediction overlay video
    """
    index_df, paths = ensure_index_for_notebook(cfg, force_rebuild=force_rebuild_index)
    matches = index_df[index_df["video_id"] == video_id]
    if len(matches) == 0:
        raise ValueError(f"{video_id} was not found in the built index. Check MAX_VIDEOS and VIDEO_DIRS.")

    row = matches.iloc[0]
    threshold = float(cfg.raw_threshold if threshold is None else threshold)

    model, ckpt_path, _ = load_trained_model_for_notebook(
        cfg,
        checkpoint_path=checkpoint_path,
        pretrained_backbone_for_load=False,
    )

    result = run_full_video_inference(
        model,
        row,
        cfg,
        raw_threshold=threshold,
        apply_postprocess=True,
        use_hysteresis=False,
    )

    labels = result["labels"].astype(int)
    raw_preds = result["raw_preds"].astype(int)
    post_preds = result["post_preds"].astype(int)
    raw_probs = result["raw_probs"].astype(float)
    smoothed_probs = result["smoothed_probs"].astype(float)

    stage2_df = pd.DataFrame({
        "frame_idx": np.arange(len(post_preds)),
        "gt_fight_label": labels,
        "created_fight_label": post_preds,
        "raw_created_fight_label": raw_preds,
        "fight_prob": raw_probs,
        "smoothed_fight_prob": smoothed_probs,
    })
    stage2_df["label_match"] = (stage2_df["gt_fight_label"] == stage2_df["created_fight_label"]).astype(int)

    save_dir = Path(save_dir) if save_dir is not None else paths.pred_dir / "single_video_created_labels"
    save_dir.mkdir(parents=True, exist_ok=True)

    side_by_side_path = save_dir / f"{video_id}_stage2_side_by_side_labels_thr_{threshold:.2f}.csv"
    label_only_path = save_dir / f"{video_id}_created_labels_only_thr_{threshold:.2f}.csv"
    summary_path = save_dir / f"{video_id}_summary_thr_{threshold:.2f}.json"

    stage2_df.to_csv(side_by_side_path, index=False)
    np.savetxt(label_only_path, post_preds, fmt="%d", delimiter=",")

    raw_metrics = compute_numpy_metrics(labels, raw_preds, raw_probs)
    post_metrics = compute_numpy_metrics(labels, post_preds, smoothed_probs)

    summary = {
        "video_id": video_id,
        "video_path": str(row["video_path"]),
        "checkpoint_path": str(ckpt_path),
        "threshold": threshold,
        "num_frames": int(len(post_preds)),
        "gt_segments": [(int(s), int(e)) for s, e in get_positive_ranges(labels)],
        "raw_pred_segments": [(int(s), int(e)) for s, e in result["raw_segments"]],
        "post_pred_segments": [(int(s), int(e)) for s, e in result["post_segments"]],
        "raw_metrics": raw_metrics,
        "post_metrics": post_metrics,
    }
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("Raw metrics:")
    print(json.dumps(raw_metrics, indent=2))
    print("\nPost-processed metrics:")
    print(json.dumps(post_metrics, indent=2))
    print("\nSaved:")
    print("Side-by-side CSV:", side_by_side_path)
    print("Created-label-only CSV:", label_only_path)
    print("Summary JSON:", summary_path)

    overlay_path = None
    if export_overlay:
        overlay_path = paths.label_video_dir / f"{video_id}_gt_plus_created_thr_{threshold:.2f}.mp4"
        export_labeled_video(
            Path(row["video_path"]),
            labels,
            overlay_path,
            cfg=cfg,
            pred_labels=post_preds,
            pred_probs=smoothed_probs,
            max_frames=cfg.label_video_max_frames,
            resize_width=cfg.label_video_resize_width,
            fps=float(row["label_fps_used"]),
            extra_text=f"created labels, thr={threshold:.2f}",
        )
        print("Overlay video:", overlay_path)

    if show_plot:
        plot_single_full_video(video_id, labels, raw_probs, smoothed_probs, post_preds, paths)

    saved_paths = {
        "side_by_side_csv": side_by_side_path,
        "created_labels_csv": label_only_path,
        "summary_json": summary_path,
        "overlay_video": overlay_path,
    }
    return stage2_df, result, saved_paths


## Configuration

Edit this cell before running. By default it builds the index and writes the final GT frame-label CSVs.

In [ ]:
# ============================================================
# Main notebook configuration
# ============================================================
# Choose one workflow:
#   "index"    -> build the video index and final GT frame-label CSVs
#   "train"    -> index + train + validation/full-video checks
#   "posttest" -> load a checkpoint and run threshold/post-test evaluation
#   "none"     -> only define functions; use the manual cells below
WORKFLOW = "index"

# Storage options.
MOUNT_DRIVE = True

# Default: use Google Drive as the dataset location.
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/CS 585")

# Local storage option. In Colab this is faster but temporary.
LOCAL_BASE_DIR = Path("/content/ntu_cctv_stage1") if Path("/content").exists() else Path.cwd() / "ntu_cctv_stage1"

# Set one of these to True when desired:
USE_LOCAL_STORAGE = False          # dataset is already in LOCAL_BASE_DIR
COPY_DRIVE_DATA_TO_LOCAL = False   # copy Drive dataset to LOCAL_BASE_DIR first
OVERWRITE_LOCAL_COPY = False

# If using local data in Colab, keep outputs on Drive so they persist after the runtime ends.
SAVE_OUTPUTS_TO_DRIVE = True

# Dataset/output paths.
OUTPUT_NAME = "ntu_cctv_stage1_outputs_combined"
CUSTOM_VIDEO_DIRS = None  # example: [Path("/path/to/original_format-001-200")]
CUSTOM_GT_JSON = None     # example: Path("/path/to/groundtruth.json")
CHECKPOINT_PATH = None    # example: DRIVE_BASE_DIR / "ntu_cctv_stage1_outputs_all_folders_fixed_labels/checkpoints/stage1_best_ntu_all_folders.pt"

# Index/label settings.
MAX_VIDEOS = None                 # None = all matched videos. For debugging: 20, 50, 200
USE_JSON_SUBSETS = True
LABEL_LENGTH_SOURCE = "video"     # "video" is safest for readable frame alignment
LABEL_FPS_SOURCE = "json"         # NTU annotation segments are in seconds; JSON FPS is usually safest
VERIFY_DECODED_FRAME_COUNT = True # True is safer; False is faster
SEQUENTIAL_READ_FALLBACK = True

# Training settings.
NUM_EPOCHS = 5
BATCH_SIZE = 8
NUM_WORKERS = 0
IMG_SIZE = 160
WINDOW_SIZE = 32
TRAIN_STRIDE = 64
INFER_STRIDE = 16
MAX_TRAIN_WINDOWS_PER_VIDEO = 8
MAX_VAL_WINDOWS_PER_VIDEO = 4
MAX_TOTAL_TRAIN_WINDOWS = 5000
MAX_TOTAL_VAL_WINDOWS = 1500
LR = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5
PATIENCE = 3
FREEZE_BACKBONE = True
PRETRAINED_BACKBONE = True

# Threshold/post-processing settings.
RAW_THRESHOLD = 0.50
THRESHOLDS_TO_PLOT = [0.30, 0.40, 0.50, 0.60, 0.70]
GLOBAL_THRESHOLDS = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]
EVAL_SPLIT = "validation"         # "training", "validation", "testing", or "all"
MAX_EVAL_VIDEOS = 20
POST_SMOOTH_K = 5
POST_HIGH_THRESH = 0.60
POST_LOW_THRESH = 0.40
POST_MAX_GAP = 5
POST_MIN_LEN = 8
POST_PRE_PAD = 5
POST_POST_PAD = 5

# Visualization/export settings.
EXPORT_GT_OVERLAY = True
EXPORT_SAMPLE_OVERLAY = True
LABEL_VIDEO_MAX_FRAMES = 600      # None for full video overlays
LABEL_VIDEO_RESIZE_WIDTH = 720
RUN_GRADCAM = False

# Build the effective paths.
if MOUNT_DRIVE:
    maybe_mount_drive(DRIVE_BASE_DIR, mount_drive=True)

if COPY_DRIVE_DATA_TO_LOCAL:
    copy_ntucctv_data_to_local(
        DRIVE_BASE_DIR,
        LOCAL_BASE_DIR,
        overwrite=OVERWRITE_LOCAL_COPY,
    )

BASE_DIR = LOCAL_BASE_DIR if (USE_LOCAL_STORAGE or COPY_DRIVE_DATA_TO_LOCAL) else DRIVE_BASE_DIR
VIDEO_DIRS = [BASE_DIR / subdir for subdir in DEFAULT_VIDEO_SUBDIRS] if CUSTOM_VIDEO_DIRS is None else [Path(p) for p in CUSTOM_VIDEO_DIRS]
GT_JSON_PATH = None if CUSTOM_GT_JSON is None else Path(CUSTOM_GT_JSON)

output_base_dir = DRIVE_BASE_DIR if (SAVE_OUTPUTS_TO_DRIVE and DRIVE_BASE_DIR.exists()) else BASE_DIR
OUTPUT_DIR = output_base_dir / OUTPUT_NAME

cfg = Config(
    command=WORKFLOW,
    base_dir=BASE_DIR,
    video_dirs=VIDEO_DIRS,
    gt_json=GT_JSON_PATH,
    output_dir=OUTPUT_DIR,
    max_videos=MAX_VIDEOS,
    use_json_subsets=USE_JSON_SUBSETS,
    label_length_source=LABEL_LENGTH_SOURCE,
    label_fps_source=LABEL_FPS_SOURCE,
    verify_decoded_frame_count=VERIFY_DECODED_FRAME_COUNT,
    sequential_read_fallback=SEQUENTIAL_READ_FALLBACK,
    img_size=IMG_SIZE,
    window_size=WINDOW_SIZE,
    train_stride=TRAIN_STRIDE,
    infer_stride=INFER_STRIDE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_train_windows_per_video=MAX_TRAIN_WINDOWS_PER_VIDEO,
    max_val_windows_per_video=MAX_VAL_WINDOWS_PER_VIDEO,
    max_total_train_windows=MAX_TOTAL_TRAIN_WINDOWS,
    max_total_val_windows=MAX_TOTAL_VAL_WINDOWS,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    patience=PATIENCE,
    freeze_backbone=FREEZE_BACKBONE,
    pretrained_backbone=PRETRAINED_BACKBONE,
    raw_threshold=RAW_THRESHOLD,
    thresholds_to_plot=THRESHOLDS_TO_PLOT,
    global_thresholds=GLOBAL_THRESHOLDS,
    eval_split=EVAL_SPLIT,
    max_eval_videos=MAX_EVAL_VIDEOS,
    post_smooth_k=POST_SMOOTH_K,
    post_high_thresh=POST_HIGH_THRESH,
    post_low_thresh=POST_LOW_THRESH,
    post_max_gap=POST_MAX_GAP,
    post_min_len=POST_MIN_LEN,
    post_pre_pad=POST_PRE_PAD,
    post_post_pad=POST_POST_PAD,
    export_gt_overlay=EXPORT_GT_OVERLAY,
    export_sample_overlay=EXPORT_SAMPLE_OVERLAY,
    label_video_max_frames=LABEL_VIDEO_MAX_FRAMES,
    label_video_resize_width=LABEL_VIDEO_RESIZE_WIDTH,
    run_gradcam=RUN_GRADCAM,
    checkpoint=None if CHECKPOINT_PATH is None else Path(CHECKPOINT_PATH),
)

# Post-test should not need ImageNet weight download when a full checkpoint is loaded.
if WORKFLOW == "posttest":
    cfg.pretrained_backbone = False
    # For faster post-testing, set this False unless specifically need decoded-frame verification.
    cfg.verify_decoded_frame_count = VERIFY_DECODED_FRAME_COUNT

set_seed(cfg.seed)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print("Workflow:", WORKFLOW)
print("Device:", cfg.device)
print("BASE_DIR:", cfg.base_dir)
print("OUTPUT_DIR:", cfg.output_dir)
print("GT_JSON:", cfg.gt_json)
print("VIDEO_DIRS:")
for d in cfg.video_dirs:
    print(" ", d, "| exists:", d.exists())
print("CHECKPOINT_PATH:", cfg.checkpoint)


## Run selected workflow

In [ ]:
# ============================================================
# Run the selected workflow
# ============================================================
# Change WORKFLOW in the configuration cell above, then run this cell.

if WORKFLOW == "index":
    run_index_only(cfg)

elif WORKFLOW == "train":
    run_train(cfg)

elif WORKFLOW == "posttest":
    run_posttest(cfg)

elif WORKFLOW in {"none", "manual", ""}:
    print("No automatic workflow selected. Use the manual cells below.")

else:
    raise ValueError(f"Unknown WORKFLOW: {WORKFLOW}")


## Optional single-video final label export


In [ ]:
# ============================================================
# Optional: create final model-generated labels for one video
# ============================================================

RUN_SINGLE_VIDEO_LABEL_EXPORT = False
SAMPLE_VIDEO_ID = "fight_0005"
SINGLE_VIDEO_THRESHOLD = 0.70
SINGLE_VIDEO_CHECKPOINT_PATH = CHECKPOINT_PATH  # or set a specific checkpoint Path(...)
SINGLE_VIDEO_SAVE_DIR = None                    # None -> OUTPUT_DIR/predictions/single_video_created_labels

if RUN_SINGLE_VIDEO_LABEL_EXPORT:
    stage2_df, single_result, saved_paths = export_single_video_created_labels(
        SAMPLE_VIDEO_ID,
        cfg,
        checkpoint_path=SINGLE_VIDEO_CHECKPOINT_PATH,
        threshold=SINGLE_VIDEO_THRESHOLD,
        save_dir=SINGLE_VIDEO_SAVE_DIR,
        force_rebuild_index=False,
        export_overlay=True,
        show_plot=True,
    )
    display(stage2_df.head())
    print(saved_paths)
else:
    print("Set RUN_SINGLE_VIDEO_LABEL_EXPORT = True to create labels for one video.")
